# 02_encode_uncovered_patents

**Objective:** Compute in-house PaECTER embeddings for the relevant Lens patents that could not be mapped to a family in the pre-computed EPO/USPTO corpus, where an English title and abstract are available.

**Inputs:** `patent_master.parquet`; `lens_to_familyid.parquet`; `startup_patent_matches.csv`.

**Outputs:** `recovered_embeddings.parquet` (`lens_id`, `embedding`).

In [ ]:
# Imports
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from sentence_transformers import SentenceTransformer

In [ ]:
# Configuration: paths, model, batch size, device
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
MASTER      = PROC / "patent_master.parquet"
MAPPING     = PROC / "lens_to_familyid.parquet"
MATCHES     = PROC / "startup_patent_matches.csv"
OUTPUT      = PROC / "recovered_embeddings.parquet"

MODEL_NAME  = "mpi-inno-comp/paecter"
BATCH_SIZE  = 16
MIN_TEXT_LEN = 10

DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available()
          else "cpu")
print(f"Device: {DEVICE}")
print(f"Modell: {MODEL_NAME}")

## Identify recoverable lens_ids

In [ ]:
# Identify relevant lens_ids without a family mapping that have usable English text
print("\n=== Identifiziere recoverable lens_ids ===")
mapping = pd.read_parquet(MAPPING, engine="fastparquet")
matches = pd.read_csv(MATCHES, usecols=["lens_id"])
unique_lens = set(matches["lens_id"].dropna().unique())
mapped_lens = set(mapping.loc[mapping["docdb_family_id"].notna(), "lens_id"])
missing = unique_lens - mapped_lens
print(f"  unique lens_ids im Match-Sample : {len(unique_lens):,}")
print(f"  davon ohne Family-Mapping       : {len(missing):,}")

pm = pd.read_parquet(MASTER, engine="fastparquet",
    columns=["lens_id","jurisdiction","title_en","abstract_en"])
miss = pm[pm["lens_id"].isin(missing)].copy()
miss["text"] = (miss["title_en"].fillna("").str.strip()
                + " "
                + miss["abstract_en"].fillna("").str.strip()).str.strip()
recoverable = miss[miss["text"].str.len() >= MIN_TEXT_LEN].copy()
print(f"  davon mit englischem Text       : {len(recoverable):,}")
print(f"  endgueltig verloren             : {len(miss) - len(recoverable):,}")

## Load model

In [ ]:
# Load the PaECTER model
print("\n=== Modell laden ===")
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.eval()
DIM = model.get_sentence_embedding_dimension()
print(f"  Dim: {DIM}, max_seq_length: {model.max_seq_length}")

## Inference

In [ ]:
# Encode the recoverable texts into normalised embeddings
print(f"\n=== Encoding {len(recoverable):,} Patente (batch={BATCH_SIZE}) ===")
texts = recoverable["text"].tolist()
embs = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
embs = embs.astype(np.float32)
print(f"  shape: {embs.shape}")

## Save

In [ ]:
# Write the recovered embeddings parquet
print("\n=== Speichern ===")
table = pa.table({
    "lens_id":   pa.array(recoverable["lens_id"].tolist(), type=pa.string()),
    "embedding": pa.array(list(embs), type=pa.list_(pa.float32())),
})
pq.write_table(table, OUTPUT, compression="zstd")
size_mb = OUTPUT.stat().st_size / 1e6
print(f"  -> {OUTPUT}  ({size_mb:.1f} MB)")

## Final coverage report

In [ ]:
# Report final embedding coverage
recovered_set = set(recoverable["lens_id"])
total_covered = mapped_lens.intersection(unique_lens) | recovered_set.intersection(unique_lens)
print("\n=== End-Coverage ===")
print(f"  via PaECTER-Parquets : {len(mapped_lens & unique_lens):,}")
print(f"  + neu berechnet      : {len(recovered_set & unique_lens):,}")
print(f"  -------------------------------")
print(f"  total                : {len(total_covered):,} / {len(unique_lens):,}  "
      f"({len(total_covered)/len(unique_lens)*100:.2f}%)")
print(f"  endgueltig verloren  : {len(unique_lens - total_covered):,}")